In [ ]:
import pandas as pd
%load_ext autoreload
%autoreload 2

from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import napari
import colorcet as cc

import dnt

spots_directory = Path(r"C:\Tracking\BlastodermAnalysis\data\spots")
embryo_overview = pd.read_excel(spots_directory / "overview.xlsx", sheet_name="Sheet1")
save_path = Path(r"C:\Tracking\BlastodermAnalysis\figures\networks")


embryo_overview = embryo_overview[embryo_overview["good"]]
included = embryo_overview["Embryo"].astype(str).tolist()
condition_map = {
    str(embryo): condition for embryo, condition in zip(embryo_overview["Embryo"], embryo_overview["condition"])
}
print(condition_map)

dnt.set_plot_style()
spots_dfs, stems = dnt.load_spots_data(spots_directory, included)

print(stems)

df = spots_dfs[0]
cycles = [10, 11, 12, 13, 14]

print(df.columns)

greens = ["#143601","#1a4301","#245501","#538d22","#73a942","#aad576"][::-1]
blues = ["#012a4a","#01497c","#2a6f97","#468faf","#89c2d9"][::-1]
reds = ["#641220","#85182a","#a71e34","#bd1f36", "#da1e37"][::-1]
oranges = ["#fbba72","#ca5310","#bb4d00","#8f250c","#691e06"]
condition_pal_map = {
    "wt": blues,
    "bcd": reds,
    "trk": greens,
}
condition_main_colors = {
    c: cmap[2] for c, cmap in condition_pal_map.items()
}

all_mmfs = {}

for k in range(len(spots_dfs)):
    stem = stems[k]
    df = spots_dfs[k]

    min_mvmt_frames, times = dnt.find_stationary_timepoints(df)
    all_mmfs[stem] = min_mvmt_frames

In [ ]:
from scipy.spatial import KDTree
import networkx as nx

df = spots_dfs[0]

cycle_track_graphs = {}
cycle_track_positions = {}

for i, cycle in enumerate([10, 11, 12, 13, 14]):

    frame = all_mmfs[stems[0]][i]

    points = df[df["frame"] == frame][["x", "y", "z"]].values
    tracks = df[df["frame"] == frame]["track_id"].values

    track_graph = nx.Graph()

    tree = KDTree(points)

    cycle_max_connection_distance = tree.query(points, k=2)[0][:, 1].max() * 1.5

    dists, ii = tree.query(points, k=8, distance_upper_bound=cycle_max_connection_distance)

    for j in range(points.shape[0]):
        first_distance = dists[j, 1]
        for k in range(1, dists.shape[1]):
            if ii[j, k] == points.shape[0]:  # no more neighbors within distance
                break
            if dists[j, k] > first_distance * 1.8:  #
                break
            track_graph.add_edge(tracks[j], tracks[ii[j, k]], weight=dists[j, k])

    cycle_track_graphs[cycle] = track_graph

    track_average_points = df[df["frame"] == frame].groupby("track_id")[["x", "y", "z"]].mean()
    cycle_track_positions[cycle] = track_average_points


In [ ]:
from collections import defaultdict

blender_edge_map = defaultdict(list)

cycle_11_tracks = set(cycle_track_graphs[11].nodes())

cycle = 14
for i, (node_a, node_b, data) in enumerate(cycle_track_graphs[cycle].edges(data=True)):
    if node_a not in cycle_11_tracks or node_b not in cycle_11_tracks:
        continue

    blender_edge_map["x"].append(cycle_track_positions[cycle].loc[node_a, "x"])
    blender_edge_map["y"].append(cycle_track_positions[cycle].loc[node_a, "y"])
    blender_edge_map["z"].append(cycle_track_positions[cycle].loc[node_a, "z"])
    blender_edge_map["x"].append(cycle_track_positions[cycle].loc[node_b, "x"])
    blender_edge_map["y"].append(cycle_track_positions[cycle].loc[node_b, "y"])
    blender_edge_map["z"].append(cycle_track_positions[cycle].loc[node_b, "z"])
    blender_edge_map["group_id"].append(i)
    blender_edge_map["group_id"].append(i)
    blender_edge_map["cycle_11_connection"].append(1 if cycle_track_graphs[11].has_edge(node_a, node_b) else 0)
    blender_edge_map["cycle_11_connection"].append(1 if cycle_track_graphs[11].has_edge(node_a, node_b) else 0)


blender_edge_df = pd.DataFrame(blender_edge_map)
blender_edge_df.to_csv(save_path / f"cycle_{cycle}_edges.csv", index=False)

In [ ]:
nx.display(cycle_track_graphs[10])
plt.show()